# Objetivo General

Implementar un pipeline de procesamiento batch para datos productivos históricos de centros de cultivo salmonero, utilizando servicios cloud en AWS como equivalente al flujo solicitado en la evaluación.

El objetivo es cargar, limpiar, transformar y respaldar información operacional relacionada con biomasa, alimentación, mortalidad, temperatura y densidad, generando una base procesada lista para análisis.

---

# 1. Caso Semestral – Problema de Negocio

El proyecto aborda un caso de análisis productivo en salmonicultura. La empresa cuenta con registros históricos semanales por centro y unidad de cultivo, incluyendo variables como especie, clase de año, biomasa inicial y final, peso promedio, alimento entregado, temperatura, densidad, mortalidad, cosecha y movimientos de peces.

El problema es que estos datos, al estar en archivos tabulares históricos, requieren limpieza, normalización y transformación antes de ser utilizados para análisis. Trabajarlos manualmente en planillas limita la escalabilidad, aumenta el riesgo de errores y dificulta obtener indicadores productivos confiables.

Por ello, se propone implementar un pipeline batch que permita cargar el dataset en una zona RAW, procesarlo mediante un notebook conectado a AWS, generar indicadores y guardar una versión procesada del dataset.

El procesamiento batch es adecuado porque se trabaja con datos históricos acumulados, no con eventos en tiempo real. Esta primera versión deja fuera el análisis de video submarino y se concentra en construir una base analítica inicial.

Como resultado esperado, se busca obtener una tabla procesada con datos limpios e indicadores como mortalidad porcentual, variación de biomasa, alimento por biomasa, densidad promedio y biomasa final por unidad de cultivo. Esto permitirá apoyar el análisis productivo y la toma de decisiones operacionales.

# 2. Ingesta y Almacenamiento – Amazon S3

En la evaluación original, esta etapa corresponde a la carga del dataset en Cloud Storage. Para esta implementación, se utiliza **Amazon S3** como servicio equivalente de almacenamiento cloud.

El dataset productivo histórico será cargado en una zona RAW dentro de un bucket S3. Esta zona conserva el archivo original sin transformaciones, permitiendo mantener trazabilidad sobre la fuente inicial de datos.

## 2.1 Dataset utilizado

El archivo utilizado corresponde a un dataset histórico de producción salmonera, con registros semanales por centro y unidad de cultivo. Incluye variables como:

- Site
- Unit
- Year
- Week
- Species
- Year class
- Open Count
- Open Biomass
- Open Weight
- Feed Weight
- Temperature
- Density Avg
- Live Days
- Fish Days
- Mortality Count
- Harvest Count
- Chip Out Count
- Chip In Count
- Close Biomass

El formato utilizado para la carga inicial será **CSV**, ya que es un formato permitido por la evaluación y facilita la revisión inicial de los datos.

## 2.2 Estructura de almacenamiento en S3

Para mantener ordenado el flujo batch, se utilizará una estructura por zonas:

```text
s3://nombre-del-bucket/
│
├── raw/
│   └── productive_data/
│       └── productive_data.csv
│
├── processed/
│   └── productive_kpi/
│
└── exports/
    └── productive_results/
```

La carpeta `raw/` almacena el archivo original.  
La carpeta `processed/` almacenará los datos transformados.  
La carpeta `exports/` se utilizará para respaldos o resultados finales descargables.

![Bucket S3 creado](../screenshots/01_s3_bucket/01_bucket_creado.png)

## 2.3 Objetivo de la ingesta

El objetivo de esta etapa es dejar el dataset disponible en almacenamiento cloud para que pueda ser leído posteriormente desde el notebook y procesado mediante Python o PySpark.

Esta etapa permite separar claramente la fuente original de los datos procesados, manteniendo una arquitectura ordenada y similar a un flujo Big Data real.

## 2.4 Comandos de carga a S3

La carga del archivo puede realizarse desde la consola web de AWS o mediante AWS CLI.

![Archivo Cargado](../screenshots/01_s3_bucket/02_raw_upload.png)

Ejemplo usando AWS CLI:

```bash
aws s3 cp productive_data.csv s3://nombre-del-bucket/raw/productive_data/productive_data.csv
```

También se puede verificar que el archivo fue cargado correctamente:

```bash
aws s3 ls s3://nombre-del-bucket/raw/productive_data/
```

La carga del dataset fue comprobada desde el notebook local utilizando credenciales temporales del laboratorio AWS.

Se realizaron las siguientes validaciones:

1. Se cargaron las credenciales desde un archivo `.env`.
2. Se verificó la identidad de AWS mediante `sts.get_caller_identity()`.
3. Se listó el contenido de la carpeta RAW en Amazon S3.
4. Se validó la existencia del archivo específico mediante `head_object()`.

Con estas comprobaciones se confirma que el archivo productivo histórico fue cargado correctamente en Amazon S3 y se encuentra disponible para las siguientes etapas del pipeline batch.

## 2.5 Consideración sobre automatización

La evaluación menciona como opción automatizar la carga mediante Cloud Functions. En esta versión AWS, el equivalente podría ser **AWS Lambda**, ejecutada al detectar nuevos archivos en S3.

Sin embargo, para esta primera versión del proyecto, la carga será realizada manualmente, ya que el foco principal está en demostrar el flujo batch: almacenamiento, lectura, transformación y respaldo del dataset procesado.

La automatización de la ingesta mediante Lambda se deja como mejora futura.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION")

S3_BUCKET = os.getenv("S3_BUCKET")
S3_RAW_PREFIX = os.getenv("S3_RAW_PREFIX")
S3_RAW_KEY = os.getenv("S3_RAW_KEY")

print("Región AWS:", AWS_DEFAULT_REGION)
print("Bucket:", S3_BUCKET)
print("Prefijo RAW:", S3_RAW_PREFIX)
print("Archivo RAW:", S3_RAW_KEY)

Región AWS: us-east-1
Bucket: bigdata-salmonicultura-fabian
Prefijo RAW: raw/productive_data/
Archivo RAW: raw/productive_data/productive_data.csv


In [2]:
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

print("Cliente S3 creado correctamente.")

Cliente S3 creado correctamente.


In [3]:
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

identity = sts.get_caller_identity()

print("Cuenta AWS:", identity["Account"])
print("ARN:", identity["Arn"])

Cuenta AWS: 601270941236
ARN: arn:aws:sts::601270941236:assumed-role/voclabs/user3277614=fabi.lecaros@duocuc.cl


In [4]:
response = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_RAW_PREFIX
)

if "Contents" in response:
    print("Archivos encontrados en S3:")
    for obj in response["Contents"]:
        print("-", obj["Key"], "| Tamaño:", obj["Size"], "bytes")
else:
    print("No se encontraron archivos en la ruta indicada.")

Archivos encontrados en S3:
- raw/productive_data/ | Tamaño: 0 bytes
- raw/productive_data/productive_data.csv | Tamaño: 100429257 bytes


In [5]:
try:
    metadata = s3.head_object(
        Bucket=S3_BUCKET,
        Key=S3_RAW_KEY
    )

    print("Archivo encontrado correctamente en S3.")
    print("Ruta:", f"s3://{S3_BUCKET}/{S3_RAW_KEY}")
    print("Tamaño:", metadata["ContentLength"], "bytes")
    print("Última modificación:", metadata["LastModified"])

except Exception as e:
    print("No se pudo encontrar el archivo.")
    print("Error:", e)

Archivo encontrado correctamente en S3.
Ruta: s3://bigdata-salmonicultura-fabian/raw/productive_data/productive_data.csv
Tamaño: 100429257 bytes
Última modificación: 2026-05-22 00:16:03+00:00


# 3. Creación de Tabla RAW – AWS Athena / Glue Data Catalog

En la pauta original, esta etapa corresponde a crear una tabla RAW en BigQuery desde Cloud Storage.  
En esta implementación equivalente en AWS, se utilizará **Amazon Athena** junto con **AWS Glue Data Catalog** para crear una tabla externa sobre el archivo CSV almacenado en Amazon S3.

La tabla RAW no transforma los datos. Su objetivo es permitir consultar el archivo original desde S3, manteniendo la lógica de una zona RAW dentro del pipeline batch.

Flujo equivalente:

```text
Amazon S3 RAW
        ↓
AWS Glue Data Catalog / Athena
        ↓
Tabla RAW consultable
```

Esta tabla será usada como punto de partida para la siguiente etapa, donde se realizará la limpieza, normalización y transformación de datos.

## 3.1 Configuración requerida

Para este paso se utilizarán las variables definidas en el archivo `.env`.

Además de las credenciales AWS ya configuradas, se agregan variables específicas para Athena:

```env
ATHENA_DATABASE=salmonicultura_raw_db
ATHENA_RAW_TABLE=productive_data_raw
ATHENA_OUTPUT=s3://bigdata-salmonicultura-fabian/exports/athena_results/
CSV_DELIMITER=;
```

La variable `ATHENA_OUTPUT` indica dónde Athena guardará internamente los resultados de sus consultas.  
La variable `CSV_DELIMITER` indica el separador del archivo CSV. En este caso se usa `;`, ya que el dataset utiliza coma decimal.

In [6]:
# Esta celda carga las variables del archivo .env y deja disponible la configuración necesaria
# para conectarse a AWS, ubicar el archivo RAW en S3 y crear la tabla RAW en Athena.

from dotenv import load_dotenv
import os
import boto3
import time

load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION")

S3_BUCKET = os.getenv("S3_BUCKET")
S3_RAW_PREFIX = os.getenv("S3_RAW_PREFIX")

ATHENA_DATABASE = os.getenv("ATHENA_DATABASE")
ATHENA_RAW_TABLE = os.getenv("ATHENA_RAW_TABLE")
ATHENA_OUTPUT = os.getenv("ATHENA_OUTPUT")
CSV_DELIMITER = os.getenv("CSV_DELIMITER", ";")

print("Región AWS:", AWS_DEFAULT_REGION)
print("Bucket S3:", S3_BUCKET)
print("Ruta RAW:", S3_RAW_PREFIX)
print("Base de datos Athena:", ATHENA_DATABASE)
print("Tabla RAW:", ATHENA_RAW_TABLE)
print("Salida Athena:", ATHENA_OUTPUT)
print("Delimitador CSV:", CSV_DELIMITER)

Región AWS: us-east-1
Bucket S3: bigdata-salmonicultura-fabian
Ruta RAW: raw/productive_data/
Base de datos Athena: salmonicultura_raw_db
Tabla RAW: productive_data_raw
Salida Athena: s3://bigdata-salmonicultura-fabian/exports/athena_results/
Delimitador CSV: ;


## 3.2 Creación del cliente de Athena

Para ejecutar consultas SQL desde el notebook local, se crea un cliente de Athena utilizando las credenciales temporales del laboratorio AWS.

Athena permitirá crear una base de datos, registrar una tabla externa y consultar el archivo CSV almacenado en Amazon S3.

In [7]:
# Esta celda crea el cliente de Athena usando las credenciales temporales del laboratorio AWS.
# Este cliente será utilizado para ejecutar las consultas SQL necesarias desde el notebook.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

print("Cliente Athena creado correctamente.")

Cliente Athena creado correctamente.


## 3.3 Función auxiliar para ejecutar consultas en Athena

Athena ejecuta las consultas de forma asíncrona.  
Por eso, se define una función auxiliar que envía una consulta SQL, espera hasta que termine y devuelve el identificador de ejecución.

Esta función se reutilizará para crear la base de datos, crear la tabla RAW y validar los resultados.

In [8]:
# Esta función ejecuta una consulta SQL en Athena, espera su finalización y valida si terminó correctamente.
# Si la consulta falla, muestra el motivo del error para facilitar la corrección.

def run_athena_query(query, database=None):
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={
            "Database": database or ATHENA_DATABASE
        },
        ResultConfiguration={
            "OutputLocation": ATHENA_OUTPUT
        }
    )

    query_execution_id = response["QueryExecutionId"]

    while True:
        result = athena.get_query_execution(
            QueryExecutionId=query_execution_id
        )

        state = result["QueryExecution"]["Status"]["State"]

        if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break

        time.sleep(2)

    if state != "SUCCEEDED":
        reason = result["QueryExecution"]["Status"].get("StateChangeReason", "Sin detalle")
        raise Exception(f"La consulta falló. Estado: {state}. Motivo: {reason}")

    return query_execution_id

## 3.4 Creación de la base de datos RAW

Se crea una base de datos en Athena para organizar las tablas asociadas a la zona RAW del pipeline.

Esta base de datos cumple un rol equivalente a la capa RAW que en la pauta original se crea en BigQuery.

In [9]:
# Esta celda crea la base de datos RAW en Athena si todavía no existe.
# La base de datos se usará para registrar la tabla externa asociada al archivo CSV en S3.

create_database_query = f"""
CREATE DATABASE IF NOT EXISTS {ATHENA_DATABASE}
"""

query_id = run_athena_query(create_database_query)

print("Base de datos RAW creada o ya existente.")
print("QueryExecutionId:", query_id)

Base de datos RAW creada o ya existente.
QueryExecutionId: d1059495-a4fb-4aef-aee8-65146249028d


## 3.5 Creación de la tabla RAW externa

A continuación, se crea una tabla externa en Athena apuntando directamente a la carpeta RAW del bucket S3.

Los campos se definen inicialmente como `string`, ya que en una capa RAW se busca conservar el dato original sin aplicar transformaciones fuertes.  
La conversión de tipos, limpieza de valores numéricos y generación de indicadores se realizará en la siguiente etapa del pipeline.

In [10]:
# Esta celda crea una tabla externa RAW en Athena sobre el archivo CSV almacenado en S3.
# La tabla apunta a la carpeta RAW, no a una copia de los datos, por lo que Athena lee directamente desde S3.

raw_s3_location = f"s3://{S3_BUCKET}/{S3_RAW_PREFIX}"

create_raw_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DATABASE}.{ATHENA_RAW_TABLE} (
    site string,
    unit string,
    year_value string,
    week string,
    species string,
    year_class string,
    open_count string,
    open_biomass string,
    open_weight string,
    feed_weight string,
    temperature string,
    density_avg string,
    live_days string,
    fish_days string,
    mortality_count string,
    harvest_count string,
    chip_out_count string,
    chip_in_count string,
    close_biomass string
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '{CSV_DELIMITER}'
STORED AS TEXTFILE
LOCATION '{raw_s3_location}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
"""

query_id = run_athena_query(create_raw_table_query)

print("Tabla RAW creada correctamente.")
print("Ubicación S3:", raw_s3_location)
print("QueryExecutionId:", query_id)

Tabla RAW creada correctamente.
Ubicación S3: s3://bigdata-salmonicultura-fabian/raw/productive_data/
QueryExecutionId: b8db24a9-7d9a-4ec7-93fc-6e93cdccd767


## 3.6 Verificación de la tabla creada

Para comprobar que la tabla RAW fue registrada correctamente, se ejecuta una consulta `SHOW TABLES` sobre la base de datos creada en Athena.

Si la tabla aparece en el resultado, significa que Athena reconoce la tabla externa asociada a la zona RAW de S3.

In [11]:
# Esta celda lista las tablas existentes en la base de datos RAW de Athena.
# Sirve para comprobar que la tabla externa fue creada correctamente.

show_tables_query = f"""
SHOW TABLES IN {ATHENA_DATABASE}
"""

query_id = run_athena_query(show_tables_query)

results = athena.get_query_results(QueryExecutionId=query_id)

print("Tablas encontradas:")
for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

Tablas encontradas:
['productive_data_raw']


## 3.7 Consulta de prueba sobre la tabla RAW

Se ejecuta una consulta simple sobre la tabla RAW para visualizar las primeras filas del dataset.

Esta validación permite comprobar que Athena puede leer correctamente el archivo CSV desde Amazon S3.

In [12]:
# Esta celda consulta las primeras 10 filas de la tabla RAW.
# Si se muestran registros del dataset, significa que Athena está leyendo correctamente el archivo desde S3.

select_raw_query = f"""
SELECT *
FROM {ATHENA_DATABASE}.{ATHENA_RAW_TABLE}
LIMIT 10
"""

query_id = run_athena_query(select_raw_query)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'unit', 'year_value', 'week', 'species', 'year_class', 'open_count', 'open_biomass', 'open_weight', 'feed_weight', 'temperature', 'density_avg', 'live_days', 'fish_days', 'mortality_count', 'harvest_count', 'chip_out_count', 'chip_in_count', 'close_biomass']
['Site_001', 'Unit_001', '2022', '2022w45', 'Coho', '2022-C1', '0', '0', '', '35', '10,825', '6,190291888', '4', '303040', '114', '', '0', '75838', '247,0301913']
['Site_001', 'Unit_001', '2022', '2022w46', 'Coho', '2022-C1', '75724', '247,0301913', '3,262244351', '85', '10,78571429', '9,151088509', '7', '529919', '39', '', '0', '0', '310,3602474']
['Site_001', 'Unit_001', '2022', '2022w47', 'Coho', '2022-C1', '75685', '310,3602474', '4,100683721', '117', '11,6', '13,12865369', '7', '529620', '42', '', '0', '0', '480,8088045']
['Site_001', 'Unit_001', '2022', '2022w48', 'Coho', '2022-C1', '75643', '480,8088045', '6,356289472', '183', '12,34285714', '20,6507471', '7', '529298', '41', '', '0', '0', '754,6475011']
['Site_001'

## 3.8 Validación de cantidad de registros

Finalmente, se ejecuta una consulta `COUNT(*)` para contar la cantidad de registros disponibles en la tabla RAW.

Esta comprobación permite verificar que Athena está reconociendo el contenido del dataset cargado en la zona RAW.

In [13]:
# Esta celda cuenta la cantidad de registros disponibles en la tabla RAW.
# El resultado permite validar que el archivo fue leído como tabla consultable desde Athena.

count_query = f"""
SELECT COUNT(*) AS total_registros
FROM {ATHENA_DATABASE}.{ATHENA_RAW_TABLE}
"""

query_id = run_athena_query(count_query)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['total_registros']
['783244']


## 3.9 Resultado de la etapa

La tabla RAW fue creada correctamente utilizando Amazon Athena y AWS Glue Data Catalog como equivalente a la tabla RAW de BigQuery solicitada en la pauta original.

La tabla externa apunta directamente al archivo CSV almacenado en la zona RAW de Amazon S3:

```text
s3://bucket/raw/productive_data/
```

Se realizaron las siguientes comprobaciones:

1. Creación de la base de datos RAW en Athena.
2. Creación de la tabla externa RAW.
3. Verificación de existencia de la tabla mediante `SHOW TABLES`.
4. Consulta de prueba con `SELECT * LIMIT 10`.
5. Conteo de registros mediante `COUNT(*)`.

Con esto, el dataset original queda disponible como tabla RAW consultable, manteniendo la trazabilidad hacia el archivo original almacenado en S3.

## 3.10 Revisión de archivos y recursos creados en AWS

Para cerrar esta etapa, se realiza una revisión de los recursos generados en AWS durante la creación de la tabla RAW.

Esta revisión permite comprobar visualmente que el pipeline dejó disponibles los elementos esperados:

1. El archivo original permanece en la zona RAW de Amazon S3.
2. Athena generó archivos de resultado en la carpeta configurada para consultas.
3. La base de datos RAW fue creada en Athena.
4. La tabla externa RAW quedó registrada y disponible para consultas.

Esta sección también sirve como punto de evidencia para agregar capturas de pantalla del laboratorio.

In [15]:
# Esta celda revisa la carpeta de salida configurada para Athena.
# Athena guarda en esta ubicación los resultados de las consultas ejecutadas desde el notebook.

athena_output_prefix = ATHENA_OUTPUT.replace(f"s3://{S3_BUCKET}/", "")

response_athena_output = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=athena_output_prefix
)

print("Archivos generados por Athena:")

if "Contents" in response_athena_output:
    for obj in response_athena_output["Contents"]:
        print("-", obj["Key"], "| Tamaño:", obj["Size"], "bytes")
else:
    print("No se encontraron archivos en la carpeta de salida de Athena.")

Archivos generados por Athena:
- exports/athena_results/38c82c06-1040-4a00-9d4c-37d94e9a304d.csv | Tamaño: 1839 bytes
- exports/athena_results/38c82c06-1040-4a00-9d4c-37d94e9a304d.csv.metadata | Tamaño: 1038 bytes
- exports/athena_results/b8db24a9-7d9a-4ec7-93fc-6e93cdccd767.txt | Tamaño: 0 bytes
- exports/athena_results/c9bb1749-3c96-4675-b9a7-00446d738460.txt | Tamaño: 19 bytes
- exports/athena_results/c9bb1749-3c96-4675-b9a7-00446d738460.txt.metadata | Tamaño: 312 bytes
- exports/athena_results/d1059495-a4fb-4aef-aee8-65146249028d.txt | Tamaño: 0 bytes
- exports/athena_results/fc3e99d0-b611-4f7e-b665-28eb13661eda.csv | Tamaño: 27 bytes
- exports/athena_results/fc3e99d0-b611-4f7e-b665-28eb13661eda.csv.metadata | Tamaño: 87 bytes


**Archivos de resultado generados por Athena**

![Resultados Athena en S3](../screenshots/01_s3_bucket/03_resultados_athena.png)

*Nota: La imagen muestra los archivos generados automáticamente por Athena al ejecutar consultas sobre la tabla RAW.*

In [16]:
# Esta celda muestra la base de datos y tabla RAW utilizadas en Athena.
# El objetivo es dejar evidencia textual dentro del notebook de los nombres usados en AWS.

print("Recurso creado en Athena / Glue Data Catalog")
print("Base de datos RAW:", ATHENA_DATABASE)
print("Tabla RAW:", ATHENA_RAW_TABLE)
print("Ubicación de datos RAW:", f"s3://{S3_BUCKET}/{S3_RAW_PREFIX}")
print("Ubicación de resultados Athena:", ATHENA_OUTPUT)

Recurso creado en Athena / Glue Data Catalog
Base de datos RAW: salmonicultura_raw_db
Tabla RAW: productive_data_raw
Ubicación de datos RAW: s3://bigdata-salmonicultura-fabian/raw/productive_data/
Ubicación de resultados Athena: s3://bigdata-salmonicultura-fabian/exports/athena_results/


**Base de datos y tabla RAW en Athena**

![Tabla RAW en Athena](../screenshots/02_athena/04_athena_tabla.png)

*Nota: La imagen muestra la tabla externa RAW registrada en Athena, disponible para consultar el archivo almacenado en S3.*

### Resultado de la revisión

La revisión confirma que la etapa de creación de tabla RAW fue completada correctamente.

Se verificó que:

- El archivo original permanece almacenado en la zona RAW de Amazon S3.
- La tabla RAW fue registrada en Athena / Glue Data Catalog.
- Athena puede consultar el archivo original desde S3.
- Las consultas ejecutadas generaron archivos de resultado en la carpeta configurada para Athena.
- Existen evidencias visuales mediante capturas de pantalla del bucket, la tabla RAW y los resultados de consulta.

Con esto se cierra la etapa equivalente a la creación de una tabla RAW en BigQuery, adaptada a la arquitectura AWS del proyecto.